In [1]:
from SleepMReye.sleepmreye.sleep_mrio import SleepMRIO

mrio = SleepMRIO(
    bids_root="/Users/zach/2025_RA/matthias/bids_sleepmreye",
    tr=2.1,
    task="sleep",
    desc_add="clean",
)




EOG data not full for 16, 1 (0/427). Skipping
EOG data not full for 16, 2 (282/427). Skipping
EOG data not full for 16, 3 (0/427). Skipping
EOG data not full for 16, 4 (234/427). Skipping
EOG data not full for 16, 5 (0/427). Skipping


In [2]:
from SleepMReye.plotting import plot_surface_searchlight
from pathlib import Path
from nilearn.image import new_img_like
from mreyemove.plotting.glm import plot_img_on_surf_with_contour, custom_cmap
from mreyemove.analysis.searchlight_correlation import run_fsaverage_surface_searchlight_correlation
from mreyemove.analysis.searchlight_correlation import build_fsaverage_neighborhoods
import nibabel as nib


# --- Run searchlight and plot ---
radius_mm = 38  # geodesic radius in mm
surf_mesh = "fsaverage7"

neighborhoods = build_fsaverage_neighborhoods(surf_mesh=surf_mesh, radius_mm=radius_mm)
corr_maps = {}
window_size = 3 


[fetch_surf_fsaverage] Dataset found in /Users/zach/nilearn_data/fsaverage
Building geodesic neighborhoods (radius=38mm)...


Dijkstra neighborhoods:   0%|          | 0/328 [00:00<?, ?it/s]

Neighbors — Mean: 2230, Median: 2164, Min: 799, Max: 4393
Building geodesic neighborhoods (radius=38mm)...


Dijkstra neighborhoods:   0%|          | 0/328 [00:00<?, ?it/s]

Neighbors — Mean: 2272, Median: 2208, Min: 865, Max: 4762


In [6]:
from SleepMReye.plotting import plot_surface_searchlight
from pathlib import Path
from nilearn.image import new_img_like
from mreyemove.plotting.glm import plot_img_on_surf_with_contour, custom_cmap
from mreyemove.analysis.searchlight_correlation import run_fsaverage_surface_searchlight_correlation
from mreyemove.analysis.searchlight_correlation import build_fsaverage_neighborhoods
import nibabel as nib

threshold = 0.3
brain_mask = nib.load(mrio.second_level_glm_mask_path()).get_fdata().astype(bool)

for state_1 in ("W",):
    print(f"=== Head Motion Controls Raw vs Clean for {state_1} ===")
    output_dir = Path("/Users/zach/2025_RA/matthias/final images/group_level/head_motion_controls")
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / f"{state_1}_correlation_map"
    data_a,_ = mrio.load_FLAME_result(contrast=f"mreyemove_{state_1}", desc_add="clean-with-fd-clamped-original-sleep-incl-nrem3-flame")
    data_b,_ = mrio.load_FLAME_result(contrast=f"mreyemove_{state_1}", desc_add="raw-original-sleep-incl-nrem3-flame")    
    
    print(data_a.t_stat.shape)

    corr_maps, _ = run_fsaverage_surface_searchlight_correlation(
        data_b.t_stat, data_a.t_stat,
        surf_mesh=surf_mesh,
        neighborhoods=neighborhoods
    )

    # for hemi in ("left", "right"):
    #     c = corr_maps[hemi]
    #     mask = np.abs(c) > 1e-6
    #     print(f"  {hemi}: mean r={c[mask].mean():.3f}, "
    #           f">0.3: {(c > 0.3).sum() / len(c)}, "
    #           f"<-0.3: {(c < -0.3).sum() / len(c)}")

    # plot_surface_searchlight(
    #     corr_maps,
    #     surf_mesh=surf_mesh,
    #     inflate=True,
    #     title=f"Searchlight r: {state_1} mreyemove vs eog ({radius_mm}mm geodesic)",
    #     output_file=f"{output_path}.png"
    #     # threshold=0.3,
    # )
    # 

=== Head Motion Controls Raw vs Clean for W ===
(53, 65, 43)
[fetch_surf_fsaverage] Dataset found in /Users/zach/nilearn_data/fsaverage
Running left hemisphere
n vertices = 163842
r_valid >=  0.3: 100.0%
r_valid <= -0.3: 0.0%
n valid vertices = 163828
r_valid >=  0.3: 100.0%
r_valid <= -0.3: 0.0%
Running right hemisphere
n vertices = 163842
r_valid >=  0.3: 99.2%
r_valid <= -0.3: 0.0%
n valid vertices = 163842
r_valid >=  0.3: 99.2%
r_valid <= -0.3: 0.0%


In [7]:
import numpy as np 

data = np.hstack((corr_maps["left"], corr_maps["right"]))
# data.shape

# data = np.random.rand(1000)  # values in [0, 1)
# bins = np.arange(-1.1, 1.1, 0.1)  # edges: 0.0, 0.1, ..., 1.0
# counts, edges = np.histogram(data, bins=bins)
p80 = np.nanpercentile(data, 50)
# plt.hist(data, bins=bins)
# plt.axvline(x=p80, color="k")
print(p80)

0.9196361601352692


In [ ]:
0.9081767201423645